# Gemini embeddings for drawings 

Parallel to `03_draw_generate_clip_embedding`, but calls `gemini-embedding-2` over the
API instead of running CLIP locally.

Produces four things:

1. **Image embeddings** for full and half-time drawings (`gemini-embedding-2`, 768-dim)
2. **Cross-modal recognizability** — cosine(drawing, "a drawing of a crab")
3. **Structured vision descriptions** from `gemini-3.6-flash`, with a `features` list
   shaped to be comparable to the hand-coded `draw_full_features` column.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, ".")  # gemini_embed.py sits next to this notebook
import gemini_embed as ge

SEA_ANIMALS = ["crab", "octopus", "seahorse", "shark", "turtle", "whale"]
DIM = 768  # we use 768 here for efficiency and comprability to CLIP without losing much informaiton, but can also try with 1024 or 3072 dims.

FULL_SUBSET = None
HALF_SUBSET = None

# inputs
HIGHRES_CSV = "../data/clean/drawing_highres_clean.csv"
HALF_CSV = "../data/embeddings/half_time_sketch_embeddings/half_time_clip_embeddings.csv"


FULL_IMAGE_ROOT = "../data/raw/mongo_output"


# Outputs
OUT_DIR = Path("../data/embeddings/gemini")
FULL_OUT = OUT_DIR / "draw_gemini_embeddings_full.csv"
HALF_OUT = OUT_DIR / "draw_gemini_embeddings_half.csv"
XMODAL_OUT = OUT_DIR / "draw_gemini_xmodal.csv"
DESC_OUT = OUT_DIR / "draw_gemini_descriptions.csv"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Cache directory 
ge.CACHE_DIR = OUT_DIR / "_cache"
print("cache:", ge.cache_stats())

cache: {'vectors': 2265, 'json': 70, 'mb': 7.3}


## Full drawings

Same category cleaning as the CLIP branch: strip the `a_`/`an_`/`this_` prefix off
`text1`, underscores to spaces, filter to sea animals.

In [ ]:
full = pd.read_csv(HIGHRES_CSV)

full["target_category"] = (
    full["text1"]
    .str.replace(r"^(a|an|this)_", "", regex=True)
    .str.replace("_", " ")
    .str.lower().str.strip()
)
full = full[full["target_category"].isin(SEA_ANIMALS)].copy()
full["participant_id"] = full["participant_id"].astype(str).str.strip()


def resolve(p, root):
    if pd.isna(p):
        return None
    p = str(p)
    if not root:
        return p
    anchor = "sketches_full_dataset/"
    i = p.find(anchor)
    if i != -1:
        return str(Path(root) / p[i:])
    return str(Path(root) / p)


full["image_path"] = full["image1"].apply(lambda p: resolve(p, FULL_IMAGE_ROOT)) #resolve image path to absolute path on disk (image1 is opaque, still a todo to clean up here!)

n_found = sum(p is not None and Path(p).exists() for p in full["image_path"])
print(f"{len(full)} rows, {n_found} PNGs found on disk")
if n_found == 0:
    print("\n!! No files resolved. Check FULL_IMAGE_ROOT against a sample path:")
    print("  ", full["image_path"].dropna().iloc[0] if len(full) else "(no rows)")
full[["participant_id", "target_category", "image_path"]].head()

638 rows, 638 PNGs found on disk


,participant_id,target_category,image_path
0,AD001,seahorse,../data/raw/mongo_output/sketches_full_dataset...
1,AD001,octopus,../data/raw/mongo_output/sketches_full_dataset...
5,AD001,crab,../data/raw/mongo_output/sketches_full_dataset...
6,AD001,turtle,../data/raw/mongo_output/sketches_full_dataset...
7,AD001,shark,../data/raw/mongo_output/sketches_full_dataset...


In [ ]:
%%time
#Run embedding generator
full["embedding"] = None
if FULL_SUBSET:
    order = full.sample(frac=1, random_state=0).index  # fixed shuffle
    embed_targets = full.loc[order[:FULL_SUBSET]]
else:
    embed_targets = full
print(f"embedding {len(embed_targets)}/{len(full)} rows this run")

full.loc[embed_targets.index, "embedding"] = pd.Series(
    ge.embed_images(embed_targets["image_path"].tolist(), dim=DIM),
    index=embed_targets.index,
)

ok = full["embedding"].notna()
print(f"embedded {ok.sum()}/{len(full)}")
full["is_adult"] = full["participant_id"].str.upper().str.startswith("AD")
print(full.loc[ok, "is_adult"].value_counts().rename({False: "child", True: "adult"}))

embedding 638/638 rows this run
  images: 638 rows, 0 missing/unreadable, 0 to fetch, 638 cached
embedded 638/638
is_adult
child    518
adult    120
Name: count, dtype: int64
CPU times: user 5.78 s, sys: 204 ms, total: 5.99 s
Wall time: 6.21 s


In [ ]:
# Vectors stored as list-strings, matching the format 04a/04c already know how to parse. 
full_save = full.loc[ok, ["participant_id", "age_months", "target_category",
                          "image_path", "is_adult", "embedding"]].copy()
full_save["embedding"] = full_save["embedding"].apply(lambda v: np.asarray(v).tolist())
full_save.to_csv(FULL_OUT, index=False)
print(f"saved {len(full_save)} rows -> {FULL_OUT}")

saved 638 rows -> ../data/embeddings/gemini/draw_gemini_embeddings_full.csv


## Half-time drawings

The half-time set is keyed by `participant_id` x `category` in
`half_time_clip_embeddings.csv`; we reuse that as the row spine and resolve image paths
from `HALF_IMAGE_PATTERN`. Non-pilot filter matches the CLIP branch: adults AD001–AD020,
children BD025+.

In [ ]:
half = pd.read_csv(HALF_CSV)
half["target_category"] = (
    half["category"].str.replace(r"^(a|an)\s+", "", regex=True).str.lower().str.strip()
)
half = half[half["target_category"].isin(SEA_ANIMALS)].copy()
half["participant_id"] = half["participant_id"].astype(str).str.strip()
half["is_adult"] = half["participant_id"].str.upper().str.startswith("AD")
half["_bd"] = pd.to_numeric(
    half["participant_id"].str.extract(r"^[Bb][Dd](\d+)$")[0], errors="coerce"
)
half = half[half["is_adult"] | (half["_bd"] >= 25)].drop(columns="_bd")


half = half.dropna(subset=["half_image_path"]).copy()
half["image_path"] = half["half_image_path"]

n_found = sum(Path(p).exists() for p in half["image_path"])
print(f"{len(half)} rows, {n_found} half-time PNGs found")
if n_found == 0:
    print("\n!! No files resolved. Check that HALF_CSV points at the right "
          "half_time_clip_embeddings.csv and that half_image_path is relative "
          "to this notebook's directory. Tried:", half["image_path"].iloc[0])

639 rows, 639 half-time PNGs found


In [ ]:
%%time
#Embed half drawings with gemini 
half["embedding"] = None
if HALF_SUBSET:
    order_h = half.sample(frac=1, random_state=0).index  # fixed shuffle
    embed_targets_h = half.loc[order_h[:HALF_SUBSET]]
else:
    embed_targets_h = half
print(f"embedding {len(embed_targets_h)}/{len(half)} rows this run")

half.loc[embed_targets_h.index, "embedding"] = pd.Series(
    ge.embed_images(embed_targets_h["image_path"].tolist(), dim=DIM),
    index=embed_targets_h.index,
)
ok_h = half["embedding"].notna()
print(f"embedded {ok_h.sum()}/{len(half)}")

# half_time_clip_embeddings.csv already carries age_months per row (same check as
# the full-drawing cell: zero mismatches against participant_age_table.csv, and
# nulls there aren't fillable from it either), so no merge needed here.
half_save = half.loc[ok_h, ["participant_id", "age_months", "target_category",
                            "image_path", "is_adult", "embedding"]].copy()
half_save["embedding"] = half_save["embedding"].apply(lambda v: np.asarray(v).tolist())
half_save.to_csv(HALF_OUT, index=False)
print(f"saved {len(half_save)} rows -> {HALF_OUT}")

embedding 639/639 rows this run
  images: 639 rows, 0 missing/unreadable, 111 to fetch, 528 cached
  [retry 1/6] embed x6: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 1000, model: gemini-embedding-2\nPlease retry in 8.702052115s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/embed_content_free_tier_requests', 'quotaId': 'EmbedContentRequ

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 1000, model: gemini-embedding-2\nPlease retry in 3.434782898s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/embed_content_free_tier_requests', 'quotaId': 'EmbedContentRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-embedding-2

## Cross-modal recognizability

Six text vectors, one per category label, compared against every drawing. `gemini_xmodal_cossim_{cat}` is how much
the drawing looks like that animal, and `gemini_xmodal_correct` is whether the intended category won.

In [7]:
labels = ge.label_embeddings(SEA_ANIMALS, template="a drawing of a {}", dim=DIM)
print("label vectors:", {k: len(v) for k, v in labels.items()})

rows = []
for _, r in full.loc[ok].iterrows():
    sims = {f"gemini_xmodal_cossim_{c}": ge.cosine(r["embedding"], labels[c])
            for c in SEA_ANIMALS}
    best = max(SEA_ANIMALS, key=lambda c: sims[f"gemini_xmodal_cossim_{c}"])
    rows.append({
        "participant_id": r["participant_id"],
        "target_category": r["target_category"],
        "age_months": r["age_months"],
        "is_adult": r["is_adult"],
        "gemini_xmodal_best": best,
        "gemini_xmodal_correct": int(best == r["target_category"]),
        **sims,
    })

xmodal = pd.DataFrame(rows)
xmodal.to_csv(XMODAL_OUT, index=False)
print(f"saved {len(xmodal)} rows -> {XMODAL_OUT}")

kids = xmodal[~xmodal["is_adult"]]
print("\nchild top-1 accuracy:", round(kids["gemini_xmodal_correct"].mean(), 3))
print("adult top-1 accuracy:",
      round(xmodal[xmodal["is_adult"]]["gemini_xmodal_correct"].mean(), 3))
print("\naccuracy by age tercile:")
print(kids.assign(bin=pd.qcut(kids["age_months"], 3))
          .groupby("bin", observed=True)["gemini_xmodal_correct"].mean().round(3))

label vectors: {'crab': 768, 'octopus': 768, 'seahorse': 768, 'shark': 768, 'turtle': 768, 'whale': 768}
saved 638 rows -> ../data/embeddings/gemini/draw_gemini_xmodal.csv

child top-1 accuracy: 0.625
adult top-1 accuracy: 0.95

accuracy by age tercile:
bin
(26.999, 68.0]    0.355
(68.0, 91.0]      0.692
(91.0, 153.0]     0.848
Name: gemini_xmodal_correct, dtype: float64


## Generate gemini descriptions

### Prompt 1: Descriptive

`gemini-3.6-flash` with a JSON schema. Slower than embedding (one call per drawing), so
run it on a subset first — `SUBSET = None` does the whole set. Cached, so building up
coverage over several runs costs nothing extra.

In [17]:
SUBSET = 50  # set to None for all drawings

targets = full.loc[ok].copy()
if SUBSET:
    targets = targets.sample(min(SUBSET, len(targets)), random_state=0)

descs = ge.describe_drawings(targets["image_path"].tolist())

desc_df = targets[["participant_id", "age_months", "target_category", "is_adult"]].copy()
desc_df["gemini_description"] = [d.get("description") if d else None for d in descs]
desc_df["gemini_features"] = [
    ",".join(d.get("features", [])) if d else None for d in descs
]
desc_df["gemini_n_features"] = [len(d.get("features", [])) if d else np.nan for d in descs]
desc_df["gemini_best_guess"] = [d.get("best_guess") if d else None for d in descs]
desc_df["gemini_confidence"] = [d.get("confidence") if d else np.nan for d in descs]
desc_df["gemini_completeness"] = [d.get("completeness") if d else None for d in descs]

desc_df.to_csv(DESC_OUT, index=False)
print(f"saved {len(desc_df)} rows -> {DESC_OUT}")
desc_df[["target_category", "gemini_best_guess", "gemini_n_features",
         "gemini_confidence", "gemini_features"]].head(10)

    described 25/50
    described 50/50
saved 50 rows -> ../data/embeddings/gemini/draw_gemini_descriptions.csv


,target_category,gemini_best_guess,gemini_n_features,gemini_confidence,gemini_features
802,crab,scribble,3,0.30,"black lines,curved strokes,diagonal lines"
918,turtle,camel,5,0.80,"long neck,head with eye and smile,two humps,tw..."
894,crab,octopus,3,0.80,"round body,two eyes,multiple legs"
1017,whale,dog,4,0.55,"round head contour,floppy ears,zig-zag mouth,l..."
143,seahorse,seahorse,4,0.95,"long snout,curled tail,head fins,eye"
282,crab,crab,5,0.95,"oval body,two claws,two eyes,smiling mouth,six..."
660,seahorse,seahorse,5,0.95,"outline drawing,snout,head,curved belly,curled..."
1052,shark,shark,6,0.95,"sharp teeth,open mouth,dorsal fin,pectoral fin..."
626,octopus,person,5,0.75,"round head,two eyes,mouth,six stick limbs,fill..."
80,whale,whale,5,0.95,"rounded body,tail fin,water spout,eye,open mouth"


## Diagnostics

To check modality and within participant spread for gemini vs CLIP

In [9]:
emb = {(r["participant_id"], r["target_category"]): r["embedding"]
       for _, r in full.loc[ok].iterrows()}

spreads = []
for pid in full.loc[ok, "participant_id"].unique():
    vecs = {c: emb[(pid, c)] for c in SEA_ANIMALS if (pid, c) in emb}
    if len(vecs) < 3:
        continue
    for c_query in vecs:
        others = [c for c in vecs if c != c_query]
        sims = [ge.cosine(vecs[c_query], vecs[c]) for c in others]
        spreads.append(max(sims) - min(sims))

spreads = np.array(spreads)
if len(spreads) == 0:
    print("Not enough coverage yet for this diagnostic: no participant has 3+ "
          "embedded categories. If you're filling in coverage via FULL_SUBSET across "
          "several runs, re-run this cell once more of the set is embedded.")
else:
    CLIP_REFERENCE_SPREAD = 0.55  # typical within-participant range in the CLIP branch
    median_spread = float(np.median(spreads))

    print(f"within-participant cosine spread (n={len(spreads)}):")
    print(f"  median {median_spread:.3f}   p10 {np.percentile(spreads, 10):.3f}"
          f"   p90 {np.percentile(spreads, 90):.3f}")
    print(f"\nsuggested softmax temperature for 04c: "
          f"{median_spread / CLIP_REFERENCE_SPREAD:.3f}")
    print("  i.e. use softmax(sims / T), or z-score sims within participant first.")

    all_sims = [ge.cosine(a, b) for a, b in zip(list(emb.values())[:-1], list(emb.values())[1:])]
    print(f"\noverall pairwise cosine: mean {np.mean(all_sims):.3f}, "
          f"min {np.min(all_sims):.3f}, max {np.max(all_sims):.3f}")
print("cache:", ge.cache_stats())

within-participant cosine spread (n=635):
  median 0.078   p10 0.039   p90 0.126

suggested softmax temperature for 04c: 0.142
  i.e. use softmax(sims / T), or z-score sims within participant first.

overall pairwise cosine: mean 0.826, min 0.702, max 0.959
cache: {'vectors': 1172, 'json': 50, 'mb': 3.8}


Conclusion: we will z-score for any cross-model comparisons

## Anatomical-parts vision descriptions (alternate prompt)

`describe_drawings()` above deliberately asks for only what's literally visible,
which is why its `gemini_features` skew toward stroke-level language ("central
scribble", "black lines") rather than the anatomical-part vocabulary human coders
used for `draw_full_features` ("shell", "legs", "pincers"). Human coders knew each
trial's intended category — that's what let them read an ambiguous blob as "shell".

`describe_drawings_anatomical()` tells Gemini the target category too, and asks for
generous, human-coder-style part inference instead of literal mark description.
Runs on the exact same 50-drawing `targets` sample as the cell above, so the two
prompt variants are directly comparable against each other and against
`draw_full_features`, not confounded by a different sample.

In [ ]:
DESC_OUT_ANATOMICAL = OUT_DIR / "draw_gemini_descriptions_anatomical.csv"

SUBSET = 50  # set to None for all drawings

targets = full.loc[ok].copy()
if SUBSET:
    targets = targets.sample(min(SUBSET, len(targets)), random_state=0)

descs_anatomical = ge.describe_drawings_anatomical(
    targets["image_path"].tolist(), targets["target_category"].tolist()
)

desc_df_anatomical = targets[["participant_id", "age_months", "target_category", "is_adult"]].copy()
desc_df_anatomical["gemini_description"] = [d.get("description") if d else None for d in descs_anatomical]
desc_df_anatomical["gemini_features"] = [
    ",".join(d.get("features", [])) if d else None for d in descs_anatomical
]
desc_df_anatomical["gemini_n_features"] = [len(d.get("features", [])) if d else np.nan for d in descs_anatomical]
desc_df_anatomical["gemini_best_guess"] = [d.get("best_guess") if d else None for d in descs_anatomical]
desc_df_anatomical["gemini_confidence"] = [d.get("confidence") if d else np.nan for d in descs_anatomical]
desc_df_anatomical["gemini_completeness"] = [d.get("completeness") if d else None for d in descs_anatomical]

desc_df_anatomical.to_csv(DESC_OUT_ANATOMICAL, index=False)
print(f"saved {len(desc_df_anatomical)} rows -> {DESC_OUT_ANATOMICAL}")
desc_df_anatomical[["target_category", "gemini_best_guess", "gemini_n_features",
                    "gemini_confidence", "gemini_features"]].head(10)

## Descriptive (1-2 word) vision descriptions (alternate prompt)

`describe_drawings_anatomical()` above asks for anatomical part names but tends to
come back with bare single words ("tentacles", "tail"). Human coders'
`draw_full_features` entries are often more descriptive two-word phrases that fold
in a count, size, or position ("eight tentacles", "long tail", "side fins").

`describe_drawings_descriptive()` uses the same anatomical prompt but explicitly
asks for one-or-two-word feature phrases, folding in a visible count/size/position
descriptor when there is one. Same output schema, same cache (keyed on prompt
text) — only the prompt differs. Runs on the same 50-drawing `targets` sample as
the cells above, so all prompt variants are directly comparable.

In [ ]:
DESC_OUT_DESCRIPTIVE = OUT_DIR / "draw_gemini_descriptions_descriptive.csv"

SUBSET = 50  # set to None for all drawings

targets = full.loc[ok].copy()
if SUBSET:
    targets = targets.sample(min(SUBSET, len(targets)), random_state=0)

descs_descriptive = ge.describe_drawings_descriptive(
    targets["image_path"].tolist(), targets["target_category"].tolist()
)

desc_df_descriptive = targets[["participant_id", "age_months", "target_category", "is_adult"]].copy()
desc_df_descriptive["gemini_description"] = [d.get("description") if d else None for d in descs_descriptive]
desc_df_descriptive["gemini_features"] = [
    ",".join(d.get("features", [])) if d else None for d in descs_descriptive
]
desc_df_descriptive["gemini_n_features"] = [len(d.get("features", [])) if d else np.nan for d in descs_descriptive]
desc_df_descriptive["gemini_best_guess"] = [d.get("best_guess") if d else None for d in descs_descriptive]
desc_df_descriptive["gemini_confidence"] = [d.get("confidence") if d else np.nan for d in descs_descriptive]
desc_df_descriptive["gemini_completeness"] = [d.get("completeness") if d else None for d in descs_descriptive]

desc_df_descriptive.to_csv(DESC_OUT_DESCRIPTIVE, index=False)
print(f"saved {len(desc_df_descriptive)} rows -> {DESC_OUT_DESCRIPTIVE}")
desc_df_descriptive[["target_category", "gemini_best_guess", "gemini_n_features",
                    "gemini_confidence", "gemini_features"]].head(10)

## Fixed coding scheme descriptions (alternate prompt)

The prompts above still leave Gemini free to invent its own feature vocabulary.
`describe_drawings_coding_scheme()` instead hands Gemini the lab's actual
per-category word bank (the same one used to code `draw_full_features` by hand —
see `ge._FEATURE_CODING_SCHEME`) and asks it to check the drawing against that
fixed list, only adding an out-of-list feature if something visible genuinely
isn't covered. This is the closest of the four prompts to "code it exactly the way
the human coders were told to." Same output schema, same cache, same 50-drawing
`targets` sample as the cells above.

In [ ]:
DESC_OUT_CODING_SCHEME = OUT_DIR / "draw_gemini_descriptions_coding_scheme.csv"

SUBSET = 50  # set to None for all drawings

targets = full.loc[ok].copy()
if SUBSET:
    targets = targets.sample(min(SUBSET, len(targets)), random_state=0)

descs_coding_scheme = ge.describe_drawings_coding_scheme(
    targets["image_path"].tolist(), targets["target_category"].tolist()
)

desc_df_coding_scheme = targets[["participant_id", "age_months", "target_category", "is_adult"]].copy()
desc_df_coding_scheme["gemini_description"] = [d.get("description") if d else None for d in descs_coding_scheme]
desc_df_coding_scheme["gemini_features"] = [
    ",".join(d.get("features", [])) if d else None for d in descs_coding_scheme
]
desc_df_coding_scheme["gemini_n_features"] = [len(d.get("features", [])) if d else np.nan for d in descs_coding_scheme]
desc_df_coding_scheme["gemini_best_guess"] = [d.get("best_guess") if d else None for d in descs_coding_scheme]
desc_df_coding_scheme["gemini_confidence"] = [d.get("confidence") if d else np.nan for d in descs_coding_scheme]
desc_df_coding_scheme["gemini_completeness"] = [d.get("completeness") if d else None for d in descs_coding_scheme]

desc_df_coding_scheme.to_csv(DESC_OUT_CODING_SCHEME, index=False)
print(f"saved {len(desc_df_coding_scheme)} rows -> {DESC_OUT_CODING_SCHEME}")
desc_df_coding_scheme[["target_category", "gemini_best_guess", "gemini_n_features",
                    "gemini_confidence", "gemini_features"]].head(10)